In [18]:
from pathlib import Path

import numpy as np
import pandas as pd
import argparse
import warnings
import h5py

from datetime import datetime, timedelta
import pagn.constants as ct

import os

In [51]:
def parse(value):
    try:
        if not '_' in value:
            return int(value)
        raise ValueError
    except ValueError:
        try:
            if not '_' in value:
                return float(value)
            raise ValueError
        except ValueError:
            try:
                if ':' in value:
                    parts = value.split(':')
                    if len(parts) == 3:
                        hours = int(parts[0])
                        minutes = int(parts[1])
                        seconds, microseconds = map(float, parts[2].split('.')) if '.' in parts[2] else (int(parts[2]), 0)
                        return timedelta(hours=hours, minutes=minutes, seconds=seconds, microseconds=int(microseconds))
                return datetime.strptime(value, "%y%m%d_%H%M")
            except ValueError:
                return value

def load_file(filename):
    params = {}
    data = {}
    
    with open(filename, 'r') as file:
        for line in file:
            if line=="Parameters:\n": continue
            elif line=="\n": continue
            elif line=="Data:\n": break
            else:
                line_splitted = line.strip().split()
                if len(line_splitted)==3: 
                    params[line_splitted[0]] = parse(line_splitted[2])
                else: 
                    params[line_splitted[0]] = []
                    for i in range(2, len(line_splitted)): 
                        params[line_splitted[0]].append(parse(line_splitted[i]))

        headers = file.readline().strip().split()
        data = {header: [] for header in headers}

        for line in file:
            values = line.strip().split()
            for i, header in enumerate(headers):
                data[header].append(parse(values[i]))
        
        for header in headers: data[header] = np.array(data[header])
    return params, data

def timescale_VRR(MSMBH, m, N, r):
    omega=(ct.G * MSMBH / (r**3))**(1/2)
    t_vrr= (2*np.pi/omega) * (MSMBH /(m*np.sqrt(N)))
    return t_vrr

In [117]:
filename='/Users/pmxks13/PhD/EMRIs_test/EMRI_Rates/Vaccaro/Rom/le_0.01/Mbh_1.0e+05/SG/alpha_0.1/spin_0.9/Tdisk_10.0/wind_On/EMRIs_G23_1g_5.txt'


params, data = load_file(Path(filename))


MSMBH=params['M_SMBH']
N=params['N']
Rg=MSMBH*ct.MSun * ct.G /(ct.c*ct.c)

for i in range(0, len(data['m1/Msun,'])):
    print(i)
    m2=data['m1/Msun,'][i]
    t_align=data['t_align/Myr,'][i]
    r=data['r0/Rg,'][i]
    
    t_vrr=timescale_VRR(MSMBH*ct.MSun, m2*ct.MSun, N, r*Rg)

    print(f'i:{i}, m:{m2} Msun, r:{r:.1f}Rg, \nt_align: {t_align:.1e}Myr, t_vrr:{t_vrr/(60*60*24*365*1e6):.1e}Myr\n')

0
i:0, m:22.67 Msun, r:7468.0Rg, 
t_align: 7.7e-15Myr, t_vrr:1.2e-05Myr

1
i:1, m:19.67 Msun, r:11600.0Rg, 
t_align: 2.6e-14Myr, t_vrr:2.6e-05Myr

2
i:2, m:18.62 Msun, r:4429.0Rg, 
t_align: 7.2e-14Myr, t_vrr:6.4e-06Myr

3
i:3, m:17.21 Msun, r:16320.0Rg, 
t_align: 1.7e-13Myr, t_vrr:4.9e-05Myr

4
i:4, m:9.335 Msun, r:9483.0Rg, 
t_align: 2.6e-13Myr, t_vrr:4.0e-05Myr

5
i:5, m:18.6 Msun, r:1903.0Rg, 
t_align: 1.5e-14Myr, t_vrr:1.8e-06Myr

6
i:6, m:15.42 Msun, r:9808.0Rg, 
t_align: 1.7e-13Myr, t_vrr:2.6e-05Myr

7
i:7, m:7.392 Msun, r:10350.0Rg, 
t_align: 4.2e-13Myr, t_vrr:5.8e-05Myr

8
i:8, m:18.6 Msun, r:1903.0Rg, 
t_align: 1.5e-14Myr, t_vrr:1.8e-06Myr

9
i:9, m:11.18 Msun, r:15670.0Rg, 
t_align: 3.7e-13Myr, t_vrr:7.1e-05Myr

10
i:10, m:6.358 Msun, r:5751.0Rg, 
t_align: 4.8e-14Myr, t_vrr:2.8e-05Myr

11
i:11, m:6.675 Msun, r:16460.0Rg, 
t_align: 6.3e-13Myr, t_vrr:1.3e-04Myr

12
i:12, m:19.41 Msun, r:5076.0Rg, 
t_align: 1.5e-14Myr, t_vrr:7.6e-06Myr

13
i:13, m:22.74 Msun, r:5465.0Rg, 
t_alig

In [134]:
k=0

for BIMF in ["Vaccaro", "Bartos", "PY", "Tagawa"]: #, "Bartos", "PY", "Tagawa", "Vaccaro"
    for RD in ["Rom", "Bartko", "PY"]: #, "Bartko", "Rom", "PY"
        for le in ["0.01"]:
            for Mbh in ['1.0e+04', '1.0e+05', '4.0e+05', '1.0e+06', '4.0e+06', '1.0e+07']: #'1.0e+04', '1.0e+05', '4.0e+05', '1.0e+06', '4.0e+06', '1.0e+07'
                for disk in ["SG"]: #, "NT"]:
                    for alpha in ["0.1"]: #, "0.01"]:
                        for spin in ["0.9"]: #, "0.99"]:
                            for Tdisk in ["10.0"]: #, "1.0", "100.0"]:
                                for wind in ["On"]: #, "Off", "Partial"]:
                                    for TT in ["G23", "P10"]: #, "P10"
                                        
                                    
                                        infile = (
                                                f"/Users/pmxks13/PhD/EMRIs_test/EMRI_Rates/"
                                                f"{BIMF}/{RD}/le_{le}/Mbh_{Mbh}/{disk}/alpha_{alpha}/spin_{spin}/"
                                                f"Tdisk_{Tdisk}/wind_{wind}/EMRIs_{TT}_1g_5.txt"
                                            )

                                        try:
                                            params, data = load_file(infile)
                                            # print(len(data['m1/Msun,']), infile)
                                            MSMBH=float(Mbh)
                                            N=1e10
                                            Rg=MSMBH*ct.MSun * ct.G /(ct.c*ct.c)

                                            if len(data)>0:
                                                for i in range(0, len(data['m1/Msun,'])):
                                                        m2=data['m1/Msun,'][i]
                                                        t_align=data['t_align/Myr,'][i]
                                                        r=data['r0/Rg,'][i]
                                                        mstar=20*ct.MSun
                                                        
                                                        t_vrr=timescale_VRR(MSMBH*ct.MSun, mstar, N, r*Rg)
        
                                                        # print(f'infile: {infile}\ni:{i}, m:{m2} Msun, r:{r:.1f}Rg, \nt_align/t_vrr: {t_align/(t_vrr/(60*60*24*365*1e6)):.1e}\n')
                                                        if 0.01*t_vrr/(60*60*24*365*1e6) < t_align:
                                                            print(f'**CANNOT NEGLECT T_VRR FOR\ninfile: {infile}\ni:{i}, M:{MSMBH:.1e} Msun, r:{r:.1f}Rg, nt_align: {t_align:.1e}Myr, t_vrr:{t_vrr/(60*60*24*365*1e6):.1e}Myr\n')
                                                            k=k+1
                                                        else:
                                                            continue
                                        except FileNotFoundError:
                                            print(f"SKIP missing event files: {infile}")
                                            continue 

                                        

print(f'no. neglected: {k}')


no. neglected: 0
